# LangChain RAG POC

POC para replicar el modulo de RAG usando documentos SST locales, chunking, recuperacion y una chain LCEL.

## Objetivo
- cargar documentos propios como `Document`
- dividirlos en chunks
- recuperar contexto relevante para una pregunta
- inyectar ese contexto en una chain LCEL
- responder con fuentes sin depender de una vector DB real en esta primera POC

In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR
for parent in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    if (parent / "src" / "sst_chatbot").exists():
        PROJECT_ROOT = parent
        break

SRC_DIR = PROJECT_ROOT / "src"
for path in (PROJECT_ROOT, SRC_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"SRC_DIR = {SRC_DIR}")

PROJECT_ROOT = c:\Users\andre\Desktop\4uentes\apps\4uentes-sst\chatboot-integration\sst_chatbot
SRC_DIR = c:\Users\andre\Desktop\4uentes\apps\4uentes-sst\chatboot-integration\sst_chatbot\src


## Capa 1: ingestion

La entrada son documentos SST normalizados. En una version real, estos documentos podrian venir de PDFs, Markdown, base de datos, articulos o actas.

In [2]:
from sst_chatbot.rag_poc import (
    RetrieverConfig,
    build_mock_rag_chat_model,
    build_rag_chain,
    build_rag_chain_with_retriever,
    build_sample_sst_documents,
    build_sst_retriever,
    format_context,
    load_source_documents,
    retrieve_relevant_documents,
    split_documents,
)

sources = build_sample_sst_documents()
documents = load_source_documents(sources)

for document in documents:
    print(document.metadata)

{'source_id': 'sst-ards-overview', 'title': 'Vision ARDS/SDD para SST', 'document_type': 'architecture_note', 'workspace_id': 'default'}
{'source_id': 'sst-rag-notes', 'title': 'RAG para agentes SST', 'document_type': 'course_note', 'workspace_id': 'default'}
{'source_id': 'sst-security-note', 'title': 'Reglas de seguridad para agentes', 'document_type': 'security_note', 'workspace_id': 'default'}


## Capa 2: indexing

Dividimos documentos en chunks para que el contexto enviado al modelo sea chico, recuperable y trazable.

In [3]:
chunks = split_documents(documents, chunk_size=220, chunk_overlap=40)

for chunk in chunks:
    print(chunk.metadata, "=>", chunk.page_content[:120])

{'source_id': 'sst-ards-overview', 'title': 'Vision ARDS/SDD para SST', 'document_type': 'architecture_note', 'workspace_id': 'default', 'chunk_index': 0} => ARDS/SDD propone que cada repositorio mantenga AGENTS.md, docs, specs y scripts de validacion. El objetivo es que humano
{'source_id': 'sst-rag-notes', 'title': 'RAG para agentes SST', 'document_type': 'course_note', 'workspace_id': 'default', 'chunk_index': 0} => RAG soluciona la falta de contexto privado en los LLMs. El flujo recomendado es cargar documentos, dividirlos en chunks,
{'source_id': 'sst-security-note', 'title': 'Reglas de seguridad para agentes', 'document_type': 'security_note', 'workspace_id': 'default', 'chunk_index': 0} => Los agentes no deben escribir archivos directamente. Deben producir intenciones estructuradas para que el backend valide


## Capa 3: retrieval

La primera POC usa recuperacion lexica simple. Mas adelante se puede reemplazar por embeddings y vector DB manteniendo el mismo contrato.

In [4]:
question = "Que aporta ARDS/SDD a los agentes SST?"
retrieved = retrieve_relevant_documents(
    question,
    chunks,
    top_k=2,
    workspace_id="default",
    search_type="mmr",
)
print(format_context(retrieved))

[source=sst-ards-overview chunk=0 score=5] Vision ARDS/SDD para SST: ARDS/SDD propone que cada repositorio mantenga AGENTS.md, docs, specs y scripts de validacion. El objetivo es que humanos y agentes trabajen con contexto versionado, portable y auditable.

[source=sst-security-note chunk=0 score=3] Reglas de seguridad para agentes: Los agentes no deben escribir archivos directamente. Deben producir intenciones estructuradas para que el backend valide rutas, permisos y contenido antes de generar artefactos descargables.


## Retriever como contrato

La app deberia depender de un retriever, no de un vector store concreto. En esta POC usamos `SSTRetriever`; mas adelante puede envolver pgvector, Elasticsearch, un vector store administrado o busqueda hibrida.

In [ ]:
retriever = build_sst_retriever(
    chunks,
    RetrieverConfig(search_type="mmr", top_k=2, fetch_k=5, workspace_id="default"),
)

for document in retriever.invoke(question):
    print(document.metadata, "=>", document.page_content[:140])

## Capa 4: answer chain

La chain usa LCEL: prepara contexto, arma prompt, invoca modelo y parsea texto. Por defecto corre en mock para no depender de cuota OpenAI.

In [6]:
USE_OPENAI = True

if USE_OPENAI:
    from sst_chatbot.config import require_env
    from sst_chatbot.langchain_poc import build_chat_model

    require_env(("OPENAI_API_KEY", "LANGCHAIN_API_KEY"))
    chat_model = build_chat_model()
    print(f"Modo actual: openai | modelo: {chat_model.model_name}")
else:
    chat_model = build_mock_rag_chat_model()
    print("Modo actual: mock | sin llamadas de red ni costo")

chain = build_rag_chain_with_retriever(chat_model, retriever)

Modo actual: openai | modelo: gpt-4.1-mini


In [7]:
answer = chain.invoke({"question": question, "workspace_id": "default"})
print(answer)

ARDS/SDD aporta a los agentes SST un marco para que cada repositorio mantenga documentación y especificaciones claras (AGENTS.md, docs, specs) junto con scripts de validación. Esto permite que tanto humanos como agentes trabajen con un contexto versionado, portable y auditable, mejorando la organización y trazabilidad. Además, bajo estas reglas, los agentes no escriben archivos directamente, sino que generan intenciones estructuradas que el backend valida en cuanto a rutas, permisos y contenido antes de crear artefactos descargables, lo que mejora la seguridad y control en la operación de los agentes.

Fuentes:
- sst-ards-overview
- sst-security-note


## Lectura para SST

RAG debe ser una capacidad opcional del agente. SST deberia enviar pregunta y contexto de seguridad; el backend recupera documentos permitidos, arma contexto y devuelve una respuesta normalizada con fuentes.